# 02 · Regresión para sugerir precio de estudios

Esta libreta ejecuta el ETL del dataset, entrena el modelo y deja el artefacto JSON que luego consume el backend al sugerir el precio de un nuevo estudio.

In [ ]:
from pathlib import Path
import csv, hashlib, json, os, subprocess

BACKEND = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'package.json').exists() and (path / 'src').exists()
)
npm = 'npm.cmd' if os.name == 'nt' else 'npm'
subprocess.run([npm, 'run', 'regression:export'], cwd=BACKEND, check=True)
subprocess.run(['python', 'ml-artifacts/scripts/regression_train.py'], cwd=BACKEND, check=True)


In [ ]:
csv_path = BACKEND / 'ml' / 'regression' / 'data' / '02_regresion_estudios_dataset.csv'
artifact_path = BACKEND / 'ml' / 'regression' / 'artifacts' / 'regression_price_model.json'
artifact = json.loads(artifact_path.read_text(encoding='utf-8'))
with csv_path.open('r', encoding='utf-8-sig', newline='') as handler:
    rows = list(csv.DictReader(handler))
csv_hash = hashlib.sha256(csv_path.read_bytes()).hexdigest()
assert csv_hash == artifact['dataset']['sha256']

print('Filas del dataset:', len(rows))
print('Artefacto usado por backend:', artifact_path)
print('Algoritmo:', artifact['algorithm'])
print('Version:', artifact['modelVersion'])
print('MAE test:', artifact['metrics']['test']['mae'])
print('R2 test:', artifact['metrics']['test']['r2'])


El backend no reentrena en cada petición: `StudyEstimationModel` carga este JSON y aplica sus coeficientes al capturar `method` y `parameter_count`.